# Exercise 07 — Advanced Claude Features

This notebook covers five advanced capabilities:

| Part | Feature | When to use |
|------|---------|-------------|
| 1 | **Extended Thinking** | Hard reasoning, maths, multi-step logic |
| 2 | **Image Input** | Analyse screenshots, photos, diagrams |
| 3 | **PDF Input** | Read and reason over documents |
| 4 | **Citations** | Verifiable answers grounded in source text |
| 5 | **Prompt Caching** | Reuse expensive context; save cost + latency |

In [ ]:
import os
import base64
import json
import time
from pathlib import Path
from dotenv import load_dotenv
import anthropic

load_dotenv(dotenv_path="../.env")
client = anthropic.Anthropic()

# Extended thinking requires claude-sonnet-3-7 or newer
THINKING_MODEL = "claude-sonnet-4-5"
MAIN_MODEL     = "claude-haiku-4-5"

print("Ready!")

---
## Part 1 — Extended Thinking

**Extended thinking** lets Claude show its work — it produces a `thinking` block (internal scratchpad) before writing its final answer.

**Why use it?**
- Improved accuracy on hard maths / logic / planning problems
- 30–50% better on benchmarks like AIME (advanced maths)

**Cost:** Thinking tokens count toward your bill, but they help Claude reason better.

**Key parameters:**
- `thinking={"type": "enabled", "budget_tokens": N}` — turn on thinking and give Claude up to N tokens to think
- `max_tokens` must be **greater than** `budget_tokens`
- Thinking blocks appear in `response.content` as `{"type": "thinking", "thinking": "..."}`

In [ ]:
# Without extended thinking
HARD_PROBLEM = """
Three friends Alice, Bob, and Carol are playing a game. Each person secretly picks a number
from 1–10. Alice's number is twice Bob's number minus 3. Carol's number is the sum of Alice's
and Bob's numbers divided by 2. The sum of all three numbers is 17. What are Alice's, Bob's,
and Carol's numbers?
"""

response_no_thinking = client.messages.create(
    model=THINKING_MODEL,
    max_tokens=512,
    messages=[{"role": "user", "content": HARD_PROBLEM}]
)

print("=== Without Extended Thinking ===")
print(response_no_thinking.content[0].text)

In [ ]:
# With extended thinking
response_with_thinking = client.messages.create(
    model=THINKING_MODEL,
    max_tokens=8000,        # must be > budget_tokens
    thinking={
        "type": "enabled",
        "budget_tokens": 5000  # max tokens Claude can use for internal reasoning
    },
    messages=[{"role": "user", "content": HARD_PROBLEM}]
)

print("=== With Extended Thinking ===")
print(f"Content blocks returned: {len(response_with_thinking.content)}")

for block in response_with_thinking.content:
    print(f"\n[Block type: {block.type}]")
    if block.type == "thinking":
        # Show just the first 500 chars of the thinking block
        print(block.thinking[:500] + ("..." if len(block.thinking) > 500 else ""))
    elif block.type == "text":
        print(block.text)

In [ ]:
# Utility: extract just the final answer text
def extract_answer(response) -> str:
    """Return only the text blocks from a response (ignores thinking blocks)."""
    return " ".join(
        block.text for block in response.content if block.type == "text"
    )


print("Final answer:")
print(extract_answer(response_with_thinking))

**Exercise 1:** Try extended thinking with a different hard problem — for example, a logic puzzle, a statistics question, or a complex planning task. Notice how the thinking block compares to the final answer.

In [ ]:
# YOUR EXERCISE 1 — change this to your hard problem
YOUR_PROBLEM = """
A snail wants to climb out of a 10m well. Each day it climbs 3m but slides back 2m at night.
How many days does it take to climb out? Show your reasoning step by step.
"""

response = client.messages.create(
    model=THINKING_MODEL,
    max_tokens=4000,
    thinking={"type": "enabled", "budget_tokens": 2000},
    messages=[{"role": "user", "content": YOUR_PROBLEM}]
)

print(extract_answer(response))

---
## Part 2 — Image Input

Claude can accept **images** alongside text. Images are passed as base64-encoded data in the `content` array.

**Supported formats:** jpeg, png, gif, webp  
**Max size per image:** 5MB  
**Max images per request:** 20

**Two ways to pass an image:**
1. **URL** — `{"type": "url", "url": "https://..."}` — Claude fetches it (public URLs only)
2. **Base64** — `{"type": "base64", "media_type": "image/png", "data": "..."}` — encode locally

In [ ]:
import struct, zlib

def create_sample_png(width: int = 40, height: int = 20) -> bytes:
    """Create a minimal valid PNG with a colour gradient (no external libs)."""
    def pack_chunk(name: bytes, data: bytes) -> bytes:
        length = struct.pack(">I", len(data))
        crc    = struct.pack(">I", zlib.crc32(name + data) & 0xFFFFFFFF)
        return length + name + data + crc

    ihdr_data = struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)
    raw_rows  = b"".join(
        b"\x00" + bytes([
            int(x / width * 255),       # R — gradient left → right
            int(y / height * 255),      # G — gradient top  → bottom
            128                         # B — constant
            for x in range(width)
        ])
        for y in range(height)
    )
    idat_data = zlib.compress(raw_rows)

    return (
        b"\x89PNG\r\n\x1a\n"
        + pack_chunk(b"IHDR", ihdr_data)
        + pack_chunk(b"IDAT", idat_data)
        + pack_chunk(b"IEND", b"")
    )


# Build a sample image and encode it
png_bytes    = create_sample_png()
image_b64    = base64.standard_b64encode(png_bytes).decode()

print(f"PNG size: {len(png_bytes)} bytes, base64 length: {len(image_b64)} chars")

# Ask Claude to describe it
response = client.messages.create(
    model=MAIN_MODEL,
    max_tokens=256,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": "image/png",
                    "data": image_b64
                }
            },
            {
                "type": "text",
                "text": "What do you see in this image? Describe the colours."
            }
        ]
    }]
)

print("\nClaude's description:")
print(response.content[0].text)

In [ ]:
# Helper to send any image file
def ask_about_image(image_path: str, question: str) -> str:
    """Load an image file and ask Claude a question about it."""
    import mimetypes
    media_type, _ = mimetypes.guess_type(image_path)
    if media_type not in ("image/jpeg", "image/png", "image/gif", "image/webp"):
        raise ValueError(f"Unsupported media type: {media_type}")

    with open(image_path, "rb") as f:
        image_data = base64.standard_b64encode(f.read()).decode()

    response = client.messages.create(
        model=MAIN_MODEL,
        max_tokens=512,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {"type": "base64", "media_type": media_type, "data": image_data}
                },
                {"type": "text", "text": question}
            ]
        }]
    )
    return response.content[0].text


print("ask_about_image() ready!")
print("Usage: ask_about_image('path/to/screenshot.png', 'What does this code do?')")

---
## Part 3 — PDF Input

Claude can read **PDF documents** directly. Send a PDF the same way as an image but use `type: "document"` and `media_type: "application/pdf"`.

**Supported models:** claude-sonnet-4-5, claude-3-5-haiku, claude-opus-4  
**Max PDF size:** 32MB (or ~100 pages)

In [ ]:
def build_minimal_pdf(text: str) -> bytes:
    """
    Generate a minimal valid PDF containing one page of text.
    No external libraries needed — builds raw PDF byte structure.
    """
    # Escape special PDF chars
    safe = text.replace('\\', '\\\\').replace('(', '\\(').replace(')', '\\)')
    # Wrap long lines
    lines = []
    for line in safe.split('\n'):
        while len(line) > 80:
            lines.append(line[:80])
            line = line[80:]
        lines.append(line)
    page_text = ') Tj\nT* ('.join(lines)

    stream = (
        f"BT\n/F1 10 Tf\n50 750 Td\n14 TL\n({page_text}) Tj\nET"
    ).encode()

    objs = {}
    body = b""
    offsets = []

    def add_obj(num: int, data: bytes):
        offsets.append(len(body))
        nonlocal body
        body += f"{num} 0 obj\n".encode() + data + b"\nendobj\n"

    add_obj(1, b"<< /Type /Catalog /Pages 2 0 R >>")
    add_obj(2, b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>")
    add_obj(3, b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Contents 4 0 R /Resources << /Font << /F1 5 0 R >> >> >>")
    stream_obj = f"<< /Length {len(stream)} >>\nstream\n".encode() + stream + b"\nendstream"
    add_obj(4, stream_obj)
    add_obj(5, b"<< /Type /Font /Subtype /Type1 /BaseFont /Courier >>")

    xref_offset = len(b"%PDF-1.4\n") + len(body)
    xref  = f"xref\n0 {len(offsets)+1}\n0000000000 65535 f \n"
    xref += "".join(f"{off + len('%PDF-1.4\n'):010d} 00000 n \n" for off in offsets)
    trailer = f"trailer\n<< /Size {len(offsets)+1} /Root 1 0 R >>\nstartxref\n{xref_offset}\n%%EOF"

    return b"%PDF-1.4\n" + body + xref.encode() + trailer.encode()


# Create a test PDF
pdf_text = """
Acme Corp Q3 Earnings Report

Revenue: $112M (up 15% YoY)
Operating Income: $28M
Net Income: $21M

Highlights:
- CloudDeploy product revenue grew 32% driven by new enterprise contracts
- Expanded into 3 new markets: Brazil, India, and Germany
- R&D investment increased to 16% of revenue
- Headcount: 3,340 (+140 vs Q2)

Risks:
- Currency headwinds reduced reported revenue by approximately 2%
- Supply chain delays continue to affect Hardware Division
"""

pdf_bytes = build_minimal_pdf(pdf_text)
pdf_b64   = base64.standard_b64encode(pdf_bytes).decode()
print(f"PDF size: {len(pdf_bytes)} bytes")

# Ask Claude about the PDF
response = client.messages.create(
    model=MAIN_MODEL,
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "base64",
                    "media_type": "application/pdf",
                    "data": pdf_b64
                }
            },
            {
                "type": "text",
                "text": "What were the Q3 revenue figures and what was the main growth driver?"
            }
        ]
    }]
)

print("\nClaude's answer from PDF:")
print(response.content[0].text)

---
## Part 4 — Citations

The **citations** feature makes Claude quote the exact part of a source document that supports each claim. This makes answers verifiable — you can trace every fact back to the source.

**How it works:**
1. Pass a document in the `content` array with `title` and `context` fields
2. Enable `citations={"enabled": True}` in the request
3. Response content blocks will include `citations` arrays with char/page locations

In [ ]:
REPORT_TEXT = """
Acme Corp 2023 Annual Report — Executive Summary

Total revenue reached $450M, a 23% increase over 2022's $366M. The Software Division
contributed $280M (62% of revenue), with flagship product CloudDeploy growing 32%.
The Hardware Division generated $95M despite supply chain disruptions causing a
6-week delay in the EdgeNode 3000 launch. Professional Services grew the fastest
at 40% YoY to reach $75M.

The engineering team resolved 847 production incidents in 2023, a significant
improvement from 1,203 incidents in 2022. The most notable incident, 2023-0042,
caused 4 hours of downtime in Q3 due to a misconfigured load balancer.

Employee headcount grew from 2,800 to 3,200. Voluntary attrition was 8.2%, below
the industry average of 13%. The company allocated $4.2M to Learning & Development,
a 35% increase, and adopted a remote-first policy for 80% of roles.
"""

response = client.messages.create(
    model=MAIN_MODEL,
    max_tokens=500,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {"type": "text", "media_type": "text/plain", "data": REPORT_TEXT},
                "title": "Acme Corp 2023 Annual Report",
                "citations": {"enabled": True}
            },
            {
                "type": "text",
                "text": "What was the revenue for each business unit, and how did the engineering team perform?"
            }
        ]
    }]
)

print("Response content blocks:")
for i, block in enumerate(response.content):
    print(f"\n[Block {i}] type={block.type}")
    if block.type == "text":
        print(f"  text: {block.text[:200]}")
        if hasattr(block, "citations") and block.citations:
            print(f"  citations ({len(block.citations)}):")
            for c in block.citations:
                c_dict = c.model_dump() if hasattr(c, "model_dump") else vars(c)
                print(f"    {json.dumps(c_dict, default=str)[:200]}")

---
## Part 5 — Prompt Caching

**Prompt caching** saves a snapshot of expensive context (system prompts, tool schemas, long documents) in Anthropic's servers between requests.

**Benefits:**
- First request creates the cache (slightly slower, normal cost)
- Subsequent requests read from cache: **~90% cost reduction** on cached tokens and **5x faster** time to first token

**How to enable:**
- Add `"cache_control": {"type": "ephemeral"}` to the content block you want cached
- The cache lasts **5 minutes** (ephemeral)
- Watch `usage.cache_creation_input_tokens` vs `usage.cache_read_input_tokens` in the response

In [ ]:
# A long system prompt — this is what we'll cache
LONG_SYSTEM_PROMPT = """
You are an expert financial analyst with 20 years of experience reading corporate reports.
You specialize in identifying key performance indicators, benchmarking against industry
standards, and providing actionable insights for investors and executives.

When answering questions:
1. Always cite specific numbers from the provided document
2. Compare figures to industry benchmarks where possible
3. Flag any concerns or red flags
4. Provide a brief investment thesis

Here is the complete Acme Corp 2023 Annual Report:

=== EXECUTIVE SUMMARY ===
Total revenue: $450M (+23% YoY). Software $280M (62%), Hardware $95M, Professional Services $75M.
EBITDA margin: 25% ($112M). R&D spend: $67M (15% of revenue). Cash: $89M.

=== SOFTWARE DIVISION ===
Products: CloudDeploy (enterprise SaaS, +32%), DataPipeline (ETL), SecureVault (encryption).
Engineering resolved 847 incidents (down from 1,203). Four major releases shipped.
Incident 2023-0042 caused 4h downtime in Q3 (misconfigured load balancer).

=== HARDWARE DIVISION ===
Revenue $95M. EdgeNode 3000 delayed 6 weeks (supply chain). Vietnam/Indonesia partnerships.

=== PROFESSIONAL SERVICES ===
Revenue $75M (+40% YoY). 312 client implementations. 94% satisfaction. GoldCare/PlatinumCare/EliteCare.

=== CYBERSECURITY ===
23 incidents, zero data breaches. ISO 27001 renewed Q1. Zero-trust architecture deployed.
Penetration testing Q1 + Q3; all critical findings remediated within 30 days.

=== HUMAN RESOURCES ===
3,200 employees (+400). Attrition 8.2% (vs 13% industry avg). Remote-first (80% of roles).
L&D budget $4.2M (+35%).
"""

print(f"System prompt length: {len(LONG_SYSTEM_PROMPT)} chars")
print("Tip: to see caching stats, call the API twice and compare usage.cache_read_input_tokens")

In [ ]:
def ask_with_cache(question: str) -> dict:
    """Ask a question using the cached system prompt, return answer + usage stats."""
    response = client.messages.create(
        model=MAIN_MODEL,
        max_tokens=512,
        system=[
            {
                "type": "text",
                "text": LONG_SYSTEM_PROMPT,
                "cache_control": {"type": "ephemeral"}  # <-- cache this block
            }
        ],
        messages=[
            {"role": "user", "content": question}
        ]
    )

    usage = response.usage
    return {
        "answer": response.content[0].text,
        "input_tokens":          usage.input_tokens,
        "output_tokens":         usage.output_tokens,
        "cache_creation_tokens": getattr(usage, "cache_creation_input_tokens", 0),
        "cache_read_tokens":     getattr(usage, "cache_read_input_tokens", 0),
    }


# First call — creates the cache
print("=== First request (creates cache) ===")
t0 = time.perf_counter()
result1 = ask_with_cache("What was Acme's EBITDA margin?")
t1 = time.perf_counter()

print(f"Answer: {result1['answer']}")
print(f"Time: {t1-t0:.2f}s")
print(f"Input tokens:          {result1['input_tokens']}")
print(f"Cache creation tokens: {result1['cache_creation_tokens']}")
print(f"Cache read tokens:     {result1['cache_read_tokens']}")

In [ ]:
# Second call — reads from cache (faster, cheaper)
print("=== Second request (reads from cache) ===")
t0 = time.perf_counter()
result2 = ask_with_cache("How many employees does Acme have, and what is the attrition rate?")
t1 = time.perf_counter()

print(f"Answer: {result2['answer']}")
print(f"Time: {t1-t0:.2f}s")
print(f"Input tokens:          {result2['input_tokens']}")
print(f"Cache creation tokens: {result2['cache_creation_tokens']}")
print(f"Cache read tokens:     {result2['cache_read_tokens']}")

print()
print("=> If cache_read_tokens > 0, the system prompt was served from cache!")

**Exercise 5:** Run ask_with_cache three times in a row with different questions and compare the `cache_read_tokens` value between calls. You should see it jump from 0 → N on the second call.

In [ ]:
# YOUR EXERCISE 5 — ask multiple questions and track cache usage
questions = [
    "What products does the Software Division sell?",
    "What were the cybersecurity highlights?",
    "Was the Hardware Division profitable?",
]

for i, q in enumerate(questions, 1):
    result = ask_with_cache(q)
    print(f"Call {i}: cache_created={result['cache_creation_tokens']} | cache_read={result['cache_read_tokens']}")
    print(f"  Q: {q}")
    print(f"  A: {result['answer'][:120]}")
    print()

---
## Summary — When to Use Each Feature

| Feature | Use when | API key |
|---------|----------|----------|
| **Extended thinking** | Multi-step math, logic puzzles, complex reasoning | `thinking={"type":"enabled","budget_tokens":N}` |
| **Image input** | Screenshots, photos, charts, diagrams | `{"type":"image","source":{"type":"base64",...}}` |
| **PDF input** | Long documents, reports, research papers | `{"type":"document","source":{"type":"base64",...}}` |
| **Citations** | Verifiable answers, fact-checking, research | `"citations":{"enabled":True}` on document block |
| **Prompt caching** | Same system prompt across many calls | `"cache_control":{"type":"ephemeral"}` on block |

---

You have now completed all 7 exercise notebooks for the **Building with the Claude API** course.

Return to [README.md](../README.md) for a full course overview.